<a href="https://colab.research.google.com/github/Rajarajan1087/fde-learning-lab/blob/FinanceFlow/FinFlow_Great_Expectations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏪 RetailCo — Great Expectations Demo
**FDE Engineering Track · Session 1 ·**

> *"The profiler tells you what is wrong. GX lets you define what 'correct' looks like
> and verify it automatically every time new data arrives.
> As an FDE you use GX to create a **data contract** — a documented, shareable set of rules
> the client's data must meet before any AI system touches it."*



### 🎯 Learner task
After the trainer runs the checkpoint, read the Data Docs report and answer:
1. Which two checks fail — and why do they block the churn model?
2. Which checks pass — what does that mean for the AI project?
3. Write one sentence for the CFO explaining the `customer_id` finding in plain language.

---

## Cell 1 — Install

In [ ]:
!pip install great-expectations --quiet
print("✅ great-expectations installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 11.5 MB/s eta 0:00:00
✅ great-expectations installed


## Cell 2 — Imports

In [ ]:
import great_expectations as gx
import pandas as pd
import os
from IPython.display import display, HTML

print(f"✅ great-expectations {gx.__version__}")

✅ great-expectations 1.18.0


## Cell 3 — Load the RetailCo dataset

The dataset is pre-loaded. This is the CRM export RetailCo handed us on Day 1.

In [ ]:
# RetailCo dataset — pre-loaded for the demo
import io

RAW_CSV = """customer_id,purchase_date,product_category,revenue,region,last_contact_date,churn_label
C1001,2023-01-15,Electronics,299.99,North,2023-01-20,0
,2023-01-17,Clothing,45.50,South,2023-01-18,
C1002,2023-01-18,Groceries,123.75,East,2023-01-25,0
C1003,2023-01-20,Electronics,899.00,West,2023-01-22,1
,2023-01-21,Clothing,67.25,North,2023-01-21,
C1004,2023-01-22,Groceries,88.40,South,2023-01-30,0
C1005,2023-01-25,Electronics,-150.00,East,2023-01-26,0
,2023-01-26,Clothing,34.99,West,2024-08-15,
C1006,2023-01-28,Groceries,210.60,North,2023-02-01,1
C1007,2023-01-30,Electronics,450.00,South,2023-02-05,0
,2023-02-01,Clothing,92.75,East,2023-02-02,
C1008,2023-02-03,Groceries,67.30,West,2023-02-10,1
C1009,2023-02-05,Electronics,1200.00,North,2023-02-08,0
,2023-02-07,Clothing,55.00,South,2023-02-07,
C1010,2023-02-10,Groceries,145.80,East,2023-02-15,0
C1001,2023-02-12,Electronics,75.00,North,2023-02-14,0
,2023-02-14,Clothing,110.25,West,2025-03-01,
C1011,2023-02-15,Groceries,88.90,North,2023-02-20,1
C1012,2023-02-18,Electronics,340.00,South,2023-02-22,0
,2023-02-20,Clothing,29.99,East,2023-02-20,
C1003,2023-02-22,Groceries,95.40,West,2023-02-25,1
C1013,2023-02-25,Electronics,-2500.00,North,2023-02-28,0
,2023-02-27,Clothing,48.75,South,2023-02-27,
C1014,2023-03-01,Groceries,175.20,East,2023-03-05,0
C1015,2023-03-03,Electronics,620.00,West,2023-03-07,1
,2023-03-05,Clothing,83.50,North,2024-11-20,
C1016,2023-03-08,Groceries,112.60,South,2023-03-12,0
C1005,2023-03-10,Electronics,245.00,East,2023-03-13,0
,2023-03-12,Clothing,61.25,West,2023-03-12,
C1017,2023-03-15,Groceries,198.40,North,2023-03-18,1
C1018,2023-03-17,Electronics,780.00,South,2023-03-20,0
,2023-03-19,Clothing,37.80,East,2023-03-19,
C1007,2023-03-22,Groceries,134.90,West,2023-03-25,0
C1019,2023-03-25,Electronics,415.00,North,2023-03-28,1
,2023-03-27,Clothing,72.40,South,2023-03-27,
C1020,2023-03-30,Groceries,89.75,East,2023-04-02,0
C1009,2023-04-01,Electronics,-85.00,West,2023-04-03,0
,2023-04-03,Clothing,56.60,North,2025-01-10,
C1021,2023-04-05,Groceries,223.30,South,2023-04-09,1
C1022,2023-04-08,Electronics,960.00,East,2023-04-11,0
,2023-04-10,Clothing,44.15,West,2023-04-10,
C1012,2023-04-13,Groceries,167.50,North,2023-04-17,0
C1023,2023-04-15,Electronics,510.00,South,2023-04-19,1
,2023-04-17,Clothing,98.90,East,2023-04-17,
C1024,2023-04-20,Groceries,76.20,West,2023-04-24,0
C1015,2023-04-22,Electronics,330.00,North,2023-04-25,1
,2023-04-24,Clothing,65.45,South,2023-04-24,
C1025,2023-04-27,Groceries,145.60,East,2023-05-01,0
C1026,2023-04-30,Electronics,875.00,West,2023-05-03,0
,2023-05-02,Clothing,52.30,North,2023-05-02,
C1017,2023-05-05,Groceries,201.40,South,2023-05-08,1
C1027,2023-05-07,Electronics,440.00,East,2023-05-11,0
,2023-05-09,Clothing,79.85,West,2024-07-04,
C1028,2023-05-12,Groceries,118.70,North,2023-05-16,1
C1019,2023-05-14,Electronics,695.00,South,2023-05-17,1
,2023-05-16,Clothing,41.20,East,2023-05-16,
C1029,2023-05-19,Groceries,88.30,West,2023-05-22,0
C1030,2023-05-21,Electronics,1150.00,North,2023-05-25,0
,2023-05-23,Clothing,107.60,South,2023-05-23,
C1021,2023-05-26,Groceries,156.80,East,2023-05-29,1
C1031,2023-05-28,Electronics,285.00,West,2023-06-01,0
,2023-05-30,Clothing,33.95,North,2025-02-14,
C1022,2023-06-02,Groceries,192.50,South,2023-06-06,0
C1032,2023-06-04,Electronics,560.00,East,2023-06-08,1
,2023-06-06,Clothing,86.75,West,2023-06-06,
C1024,2023-06-09,Groceries,103.40,North,2023-06-13,0
C1033,2023-06-11,Electronics,-320.00,South,2023-06-14,0
,2023-06-13,Clothing,58.20,East,2023-06-13,
C1025,2023-06-16,Groceries,234.60,West,2023-06-20,0
C1034,2023-06-18,Electronics,720.00,North,2023-06-22,1
,2023-06-20,Clothing,45.90,South,2024-09-30,
C1026,2023-06-23,Groceries,139.20,East,2023-06-27,0
C1035,2023-06-25,Electronics,395.00,West,2023-06-29,0
,2023-06-27,Clothing,71.35,North,2023-06-27,
C1027,2023-06-30,Groceries,178.90,South,2023-07-04,0
C1036,2023-07-02,Electronics,840.00,East,2023-07-06,1
,2023-07-04,Clothing,94.60,West,2023-07-04,
C1028,2023-07-07,Groceries,116.30,North,2023-07-11,1
C1029,2023-07-09,Electronics,475.00,South,2023-07-13,0
,2023-07-11,Clothing,62.45,East,2025-04-01,
C1037,2023-07-14,Groceries,205.70,West,2023-07-18,0
C1038,2023-07-16,Electronics,930.00,North,2023-07-20,1
,2023-07-18,Clothing,38.80,South,2023-07-18,
C1030,2023-07-21,Groceries,151.40,East,2023-07-25,0
C1039,2023-07-23,Electronics,265.00,West,2023-07-27,0
,2023-07-25,Clothing,83.15,North,2023-07-25,
C1031,2023-07-28,Groceries,197.80,South,2023-08-01,0
C1040,2023-07-30,Electronics,610.00,East,2023-08-03,1
,2023-08-01,Clothing,49.55,West,2024-12-25,
"""

df = pd.read_csv(io.StringIO(RAW_CSV))

print("✅ RetailCo dataset loaded")
print(f"   Rows    : {len(df):,}")
print(f"   Columns : {list(df.columns)}")
print()
df.head(8)

✅ RetailCo dataset loaded
   Rows    : 89
   Columns : ['customer_id', 'purchase_date', 'product_category', 'revenue', 'region', 'last_contact_date', 'churn_label']



,customer_id,purchase_date,product_category,revenue,region,last_contact_date,churn_label
0,C1001,2023-01-15,Electronics,299.99,North,2023-01-20,0.0
1,NaN,2023-01-17,Clothing,45.50,South,2023-01-18,NaN
2,C1002,2023-01-18,Groceries,123.75,East,2023-01-25,0.0
3,C1003,2023-01-20,Electronics,899.00,West,2023-01-22,1.0
4,NaN,2023-01-21,Clothing,67.25,North,2023-01-21,NaN
5,C1004,2023-01-22,Groceries,88.40,South,2023-01-30,0.0
6,C1005,2023-01-25,Electronics,-150.00,East,2023-01-26,0.0
7,NaN,2023-01-26,Clothing,34.99,West,2024-08-15,NaN


## Cell 4 — Create GX context

In [ ]:
context = gx.get_context(mode="file")
print("GX context ready")

GX context ready


## Cell 5 — Register dataset + build pre-built Expectation Suite

These are the 5 rules already written for the RetailCo data.  
**Learners:** read each rule and the comment explaining *why* it matters for the churn model.

In [ ]:
# --- Data asset ---
data_source      = context.data_sources.add_pandas("retailco_datasource")
data_asset       = data_source.add_dataframe_asset(name="retailco_sales")
batch_definition = data_asset.add_batch_definition_whole_dataframe("full_batch")

# --- Pre-built Expectation Suite ---
suite = context.suites.add(gx.ExpectationSuite(name="retailco_suite"))

# customer_id: expect not null
# A churn model tracks a customer's behaviour over time.
# If we can't identify the customer, we can't track their behaviour.
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(column="customer_id")
)

# churn_label: expect not null
# This is the target variable. A churn model cannot be trained
# if 31% of the labels are missing.
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(column="churn_label")
)

# revenue: expect values >= -1000  (reasonable refund floor)
# Legitimate returns or a data entry error — we need to find out.
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="revenue",
        min_value=-1000,
        max_value=None
    )
)

# purchase_date: expect not null
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(column="purchase_date")
)

# last_contact_date: expect not null
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(column="last_contact_date")
)

print("✅ Pre-built checkpoint ready — 5 expectations loaded")

✅ Pre-built checkpoint ready — 5 expectations loaded


## Cell 6 — Run the checkpoint

> **Trainer:** run this cell live. The results will show which rules pass and which fail.

In [ ]:
validation_definition = context.validation_definitions.add(
    gx.ValidationDefinition(
        name="retailco_validation",
        data=batch_definition,
        suite=suite,
    )
)

checkpoint = context.checkpoints.add(
    gx.Checkpoint(
        name="retailco_checkpoint",
        validation_definitions=[validation_definition],
    )
)

results = checkpoint.run(batch_parameters={"dataframe": df})

print(results)

Calculating Metrics:   0%|          | 0/30 [00:00<?, ?it/s]

run_id={"run_name": null, "run_time": "2026-06-06T07:58:43.774055+00:00"} run_results={ValidationResultIdentifier::retailco_suite/__none__/20260606T075843.774055Z/retailco_datasource-retailco_sales: {
  "success": false,
  "results": [
    {
      "success": false,
      "expectation_config": {
        "type": "expect_column_values_to_not_be_null",
        "kwargs": {
          "batch_id": "retailco_datasource-retailco_sales",
          "column": "customer_id"
        },
        "meta": {},
        "id": "0db4203a-0a66-4927-b289-ba9f440c4482",
        "severity": "critical"
      },
      "result": {
        "element_count": 89,
        "unexpected_count": 30,
        "unexpected_percent": 33.70786516853933,
        "partial_unexpected_list": [
          null,
          null,
          null,
          null,
          null,
          null,
          null,
          null,
          null,
          null,
          null,
          null,
          null,
          null,
          null,
     

## Cell 7 — Read the Data Docs report

> *"This report is not for you — it is for the client.*  
> *You email this to the Head of Sales before the 10 AM meeting.*  
> *It is evidence, not opinion."*

**Learners:** read the report below. Two checks fail, three pass.  
Which two failures directly block the AI use case — and why?

In [ ]:
context.build_data_docs()

# Find and display the validation report inline
docs_root = "gx/uncommitted/data_docs/local_site/validations"
html_file = None

for root, dirs, filenames in os.walk(docs_root):
    for f in filenames:
        if f.endswith(".html"):
            html_file = os.path.join(root, f)

if html_file:
    with open(html_file, "r") as f:
        html_content = f.read()
    print(f"📄 Showing: {html_file}\n")
    display(HTML(html_content))
else:
    print("⚠️  Data Docs not found. Make sure Cell 6 ran successfully.")

📄 Showing: gx/uncommitted/data_docs/local_site/validations/retailco_suite/__none__/20260606T075843.774055Z/retailco_datasource-retailco_sales.html



,
Evaluated Expectations,5
Successful Expectations,2
Unsuccessful Expectations,3
Success Percent,40%
,
Great Expectations Version,1.18.0
Run Name,__none__
Run Time,2026-06-06T07:58:43Z
,
ge_load_time,20260606T075843.786424Z


## Cell 8 — Download the Data Docs report

Download the report to share with the client before the 10 AM meeting.

In [ ]:
from google.colab import files

docs_root = "gx/uncommitted/data_docs/local_site/validations"
html_file = None

for root, dirs, filenames in os.walk(docs_root):
    for f in filenames:
        if f.endswith(".html"):
            html_file = os.path.join(root, f)

if html_file:
    files.download(html_file)
    print("✅ Data Docs report downloaded — send this to the client before 10 AM")
else:
    print("⚠️  No report found. Run Cell 7 first.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Data Docs report downloaded — send this to the client before 10 AM



---
## 📝 Learner Discussion Questions

After reading the Data Docs report, discuss with the class:

**1.** Two critical checks fail — customer identity and churn label.  
In your own words, why does each one block the AI use case?

**2.** Three checks pass. What does that mean for the project — is it cancelled?

**3.** Write one sentence for the CFO explaining the `customer_id` finding.  
Rules: no technical terms, no percentages, must include a business consequence.

> ❌ *"44% null rate on customer_id"*  
> ✅ *"We cannot attribute nearly half of your sales transactions to a specific customer"*
